<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [6]</a>'.</span>

# Coaxial Cable

A coaxial waveguide: an annular dielectric region (outer cylinder with inner cylinder removed). The cylinder walls (inner and outer conductors) are PEC, and the top and bottom annular faces are waveports.

We build the geometry with a boolean cut, use **selectors** to pick the two end faces, define entities for the dielectric volume and the two waveport surfaces, then generate the mesh, write the Palace config, run the simulation, and plot the S parameters.

In [1]:
import gmsh

from palacetoolkit.geometry import Selector, splitCylinderSeam
from palacetoolkit.mesh import (
    Entity,
    run_entity_pipeline,
    create_graded_mesh,
    generate_3d_mesh,
)
from palacetoolkit.simulation import (
    generate_palace_config_from_entities,
    run_palace,
)
from palacetoolkit.postpro import s_params
from palacetoolkit.viz import view_mesh

## Geometry parameters

A short coaxial cable segment. The inner conductor radius `r_inner`, outer conductor radius `r_outer`, and length `length` define the annular dielectric region.

In [2]:
r_inner = 0.5e-3   # inner conductor radius  [m]
r_outer = 1.5e-3   # outer conductor radius  [m]
length  = 20.0e-3  # coax length along z     [m]

# Frequency sweep
freq_min  = 0.1e9   # 0.1 GHz
freq_max  = 5.0e9   # 5 GHz
freq_step = 0.1e9   # 100 MHz steps

## Build the annular geometry

Create the outer cylinder and cut the inner cylinder from it to produce the annular dielectric region. Then call `splitCylinderSeam` to fragment the cylindrical surfaces at the OCC seam, preventing duplicated facets during 3D meshing.

In [3]:
gmsh.initialize()
gmsh.option.setNumber("General.Terminal", 1)
gmsh.model.add("coax")

# Outer and inner cylinders
outer = gmsh.model.occ.addCylinder(0, 0, 0, 0, 0, length, r_outer)
inner = gmsh.model.occ.addCylinder(0, 0, 0, 0, 0, length, r_inner)

# Cut: outer minus inner → annular volume
annular, _ = gmsh.model.occ.cut([(3, outer)], [(3, inner)], removeObject=True, removeTool=True)
gmsh.model.occ.synchronize()

annular_tags = [t for d, t in annular if d == 3]
print(f"Annular volume tags: {annular_tags}")

# Split the OCC cylinder seam to prevent duplicated facets during meshing
annular_tags[0] = splitCylinderSeam(annular_tags[0], 0, 0, 0, 0, 0, length, r_outer)
gmsh.model.occ.synchronize()
print(f"After seam split: {annular_tags}")

Info    : [  0%] Difference                                                                                  
Info    : [ 10%] Difference                                                                                  
Info    : [ 20%] Difference                                                                                  
Info    : [ 30%] Difference                                                                                  
Info    : [ 40%] Difference                                                                                  
Info    : [ 50%] Difference                                                                                  
Info    : [ 60%] Difference                                                                                  
Info    : [ 70%] Difference - Filling splits of edges                                                                                
Info    : [ 80%] Difference - Making faces                                                      

## Select the two waveport faces

Using the `Selector` API bound to the annular volume — pick the faces at `z=0` and `z=length`. Each end face is an annulus (a ring-shaped surface).

In [4]:
# Bind the selector to the annular volume — only its boundary faces are considered.
# This mirrors build123d/cadquery where selectors operate on a specific shape.
sel = Selector((3, annular_tags[0]))

# cadquery-style: faces at a given coordinate
# The seam split may fragment the end faces into multiple pieces, so we
# collect all faces at each z and combine their tags.
port1_faces = sel.faces_at_z(0.0)
port2_faces = sel.faces_at_z(length)

assert len(port1_faces) >= 1, f"Expected faces at z=0, got {len(port1_faces)}"
assert len(port2_faces) >= 1, f"Expected faces at z=length, got {len(port2_faces)}"

port1_tags = [f.tag for f in port1_faces]
port2_tags = [f.tag for f in port2_faces]

print(f"Port 1 (z=0):      tags={port1_tags}")
print(f"Port 2 (z=length): tags={port2_tags}")

Port 1 (z=0):      tags=[1, 6]
Port 2 (z=length): tags=[5]


## Define entities

The annular volume is a **dielectric** (vacuum, εᵣ=1). The two end faces are waveports. The remaining exterior surfaces (inner and outer cylinder walls) are PEC — they appear as `dielectric__None` in the physical group map and must be explicitly claimed.

In [5]:
entities = [
    Entity(
        name="dielectric",
        dim=3,
        btype="dielectric",
        mesh_order=0,
        tags=annular_tags,
        eps_r=1.0,
        mu_r=1.0,
        loss_tan=0.0,
    ),
    Entity(
        name="waveport_1",
        dim=2,
        btype="waveport",
        mesh_order=1,
        tags=port1_tags,
        mode=1,
        excitation=True,
    ),
    Entity(
        name="waveport_2",
        dim=2,
        btype="waveport",
        mesh_order=2,
        tags=port2_tags,
        mode=1,
        excitation=False,
    ),
    # PEC walls — the exterior surfaces of the dielectric (inner & outer conductors)
    Entity(
        name="dielectric__None",
        dim=2,
        btype="pec",
        mesh_order=3,
        tags=[],
    ),
]

pg_map = run_entity_pipeline(entities)
print("Physical group map:", pg_map)

Info    : [  0%] Difference                                                                                  
Info    : [ 10%] Difference                                                                                  
Info    : [ 20%] Difference                                                                                  
Info    : [ 30%] Difference                                                                                  
Info    : [ 40%] Difference - Repeat intersection                                                                                
Info    : [ 50%] Difference                                                                                  
Info    : [ 60%] Difference                                                                                  
Info    : [ 70%] Difference                                                                                  
Info    : [ 80%] Difference - Splitting faces                                                       

## Mesh generation

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [6]:
mesh_file = "coax.msh"

# Operating wavelength at the highest frequency (5 GHz)
c = 3e8
wavelength = c / freq_max

# Graded mesh: refine near geometric curves
create_graded_mesh(
    wavelength=wavelength,
    ppw_near=6,
    ppw_far=3,
)

generate_3d_mesh(entities, output_file=mesh_file)
print(f"Mesh written to {mesh_file}")

  global: 16 curves, SizeMin=0.0100
  ppw_near=6  ppw_far=3
  SizeMax=0.0200  transition=0.0150
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Circle)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 20%] Meshing curve 4 (Circle)
Info    : [ 30%] Meshing curve 5 (Line)
Info    : [ 40%] Meshing curve 6 (Line)
Info    : [ 40%] Meshing curve 7 (Line)
Info    : [ 50%] Meshing curve 8 (Line)
Info    : [ 60%] Meshing curve 9 (Line)
Info    : [ 60%] Meshing curve 10 (Line)
Info    : [ 70%] Meshing curve 11 (Line)
Info    : [ 70%] Meshing curve 12 (Circle)
Info    : [ 80%] Meshing curve 13 (Circle)
Info    : [ 90%] Meshing curve 14 (Circle)
Info    : [ 90%] Meshing curve 15 (Circle)
Info    : [100%] Meshing curve 16 (Line)
Info    : Done meshing 1D (Wall 0.0165672s, CPU 0.015632s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 1 (Plane, MeshAdapt)
Info    : [ 20%] Meshing surface 2 (Plane, MeshAdapt)
Info    : [ 30%] Meshing surfa

Error   : PLC Error:  A segment and a facet intersect at point


Exception: PLC Error:  A segment and a facet intersect at point

In [ ]:
view_mesh(mesh_file)

## Palace configuration & simulation

Build the Palace JSON config from the entity definitions and run the driven simulation.

In [ ]:
config_path = "coax.json"

config = generate_palace_config_from_entities(
    entity_defs=[e.to_dict() for e in entities],
    pg_map=pg_map,
    mesh_file=mesh_file,
    output_file=config_path,
    freq_min=freq_min,
    freq_max=freq_max,
    freq_step=freq_step,
    L0=1e-3,
    solver_order=2,
)

run_palace(config_path)

## S parameters

In [ ]:
csv_file = "postpro/coax/port-S.csv"

resonant_freq = s_params(csv_file)
print(f"Resonant frequency (min |S11|): {resonant_freq:.3f} GHz")